# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace
import importlib.util
import sys
# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
from typing import List, Dict, Any
from dotenv import load_dotenv
from lib.agents import UdaPlayAgent
from lib.llm import LLM  # noqa: E402
from lib.state_machine import Run
from lib.messages import BaseMessage,UserMessage,AIMessage,ToolMessage
from lib.tooling import tool
from lib.vector_db import VectorStore
from lib.evaluation import AgentEvaluator2, EvaluationReport
from chromadb.utils import embedding_functions
from tavily import TavilyClient
from dotenv import load_dotenv

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\MarkS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY"),
    api_base=os.getenv("OPENAI_BASE_URL")
)

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [4]:
import chromadb
chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("udaplay_project",embedding_function=embedding_fn)
vector_store = VectorStore(collection)

@tool
def retrieve_game(query: str) -> List[str]:
    """
    Search the vector database for video game information.

    Returns a list of matched game records, each a string formatted as:
    "[Platform - Genre] Name (Year) - Description"
    e.g. "[PlayStation 5 - Action RPG] Elden Ring (2022) - An open-world ..."

    Args:
        query (str): A question about the game industry.
    """
    results = vector_store.query(query_texts=[query], n_results=3)
    print("Retrieving documents")
    return results["documents"][0] if results["documents"] else []

#### Evaluate Retrieval Tool

In [5]:
@tool
def evaluate_retrieval(question: str, retrieve_game_output: List[str] = None) -> EvaluationReport:
    """Judge whether the most recent retrieved documents - from a vector database or web search - can sufficiently answer the question.
    The documents are supplied automatically; only pass the question.

    Args:
        question: the user's original question
       
    Returns:
        EvaluationReport:
            - usefulness: bool, whether the retrieved documents are sufficient
            - explanation: str, explanation of the evaluation
    """
    evaluator = AgentEvaluator2()
    print("Evaluating retrieved documents")
    retrieved_doc_eval: EvaluationReport = evaluator.evaluate_final_response(question,retrieve_game_output or [])
    return retrieved_doc_eval

#### Game Web Search Tool

In [6]:
@tool
def game_web_search(query: str,max_results: int = 3) -> List[str]:
    """
    Search the web for information about video games when the local vector
    database does not contain a confident answer. Returns the same structure
    as retrieve_game:
    - documents: a list of strings, each a web source formatted as
      "Title: content excerpt (source: url)"
    
    Args:
        query (str): A question about the game industry.
        max_results (int, optional): Number of web sources to draw from. Defaults to 3.
    """
    api_key = os.getenv("TAVILY_API_KEY")
    client = TavilyClient(api_key=api_key)
    result = client.search(query,search_depth='advanced',
                           include_answer=False,
                           max_results = max_results,
                           )
    
    documents = []
    for source in result.get("results",[]):
        content = source.get("content","")
        title = source.get("title","")
        url = source.get("url","")
        documents.append(f"{title}: {content} (source: {url})")
    print("Web searching")
    return documents

### Agent

In [7]:
tools = [retrieve_game,evaluate_retrieval,game_web_search]
# TODO: Create your Agent abstraction using StateMachine
udaplay_agent = UdaPlayAgent(
    model_name="gpt-4o-mini",
    instructions=f"""
        You are UdaPlay, an AI research agent and expert in video game knowledge.
        You answer questions about game names, platforms, genres, publishers, descriptions,
        and release years by calling tools and synthesizing across sources.
        
        How to work:
        - Think briefly about what the question needs, then CALL the appropriate tool.
          Use the tool-calling interface directly — do not write the action as text.
        
        - Sequence:
            1. Call retrieve_game with the user's question.
            2. Call evaluate_retrieval (pass only the question) to judge if results are sufficient.
            3. If usefulness is False, call game_web_search, then evaluate_retrieval again.
            4. If still not useful, call game_web_search again with a larger max_results.
            - When the evaluated documents are useful, STOP calling tools and write the final
            answer to the user as a well structured response that:
                - Cite information sources
                - Combine information from multiple sources when needed
                - Present information in a natural, readable format
        """,
    tools=tools,
)

In [8]:
# TODO: Invoke your agent
import json
from pathlib import Path
def print_and_record_messages(messages: List[BaseMessage],file):
    "Print and record UdaPlay output to text file"
    for m in messages:
        if isinstance(m,UserMessage):
            print("_"*50)
            print(f"User: {m.content}\n")
            #write output to text file
            file.write("_"*50)
            file.write(f"User: {m.content}\n")
        elif isinstance(m,ToolMessage):
            match m.name:
                case "evaluate_retrieval":
                    try:          
                        data=json.loads(m.content)
                        useful = "Useful" if data["usefulness"] else "Not Useful"
                        print(f"Retrieved documents are {useful}.\n{data['explanation']}.\n")
                        file.write(f"Retrieved documents are {useful}.\n{data['explanation']}.\n")
                    except (json.JSONDecodeError, KeyError, TypeError):
                        print(f"Evaluation: {m.content}\n")
                        file.write(f"Evaluation: {m.content}\n")
                        
                case "retrieve_game":
                    print("Retrieving documents from vector store\n")
                    file.write("Retrieving documents from vector store\n")
                case "game_web_search":
                    print("Retrieving documents from web search\n")
                    file.write("Retrieving documents from web search\n")
        elif isinstance(m,AIMessage) and m.content is not None:
            print(m.content)
            file.write(m.content+ "\n")
        else:
            print("\n")
            file.write("\n")
    
session_ids = ["gamer_1","gamer_2"]         
queries1 = ["When Pokémon Gold and Silver was released?",
        "Which one was the first 3D platformer Mario game?",
        "Was Mortal Kombat X realeased for Playstation 5?",
        "Who developed FIFA 21?",
        ]
queries2 = [
    "What year was The Legend of Zelda: Breath of the Wild released?",
    "Which company published Halo 3?",
    "Was Red Dead Redemption 2 available on Nintendo Switch?",
    "What platform was Pokémon Red launched on?",
    "What is Rockstar Games working on right now?"
]
query_sets = [queries1,queries2]
for session_id,queries in zip(session_ids,query_sets):
    out_path = Path(os.getcwd()) / f"{session_id}_udaplay_output.txt"
    with open(out_path,"w",encoding = "utf-8") as file:
        file.write(f"User's {session_id} session with UdaPlay\n\n")
        for i,query in enumerate(queries):
            run:Run = udaplay_agent.invoke(query, session_id)
            messages = run.get_final_state()["messages"]
            start = 0
            if i > 0:
                for j, m in enumerate(messages):
                    if isinstance(m, UserMessage):
                        start = j
                print("Omitting history\n")
                file.write("Omitting history\n")
            print_and_record_messages(messages[start:],file)
        

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
Retrieving documents
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
Evaluating retrieved documents
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
Web searching
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
Evaluating retrieved documents
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__


__________________________________________________
User: When Pokémon Gold and Silver was released?



Retrieving documents from vector store



Retrieved documents are Not Useful.
The retrieved documents do not provide a clear answer to the user's query about the release date of Pokémon Gold and Silver. While one document mentions that Pokémon Gold and Silver were relea

In [9]:
i=0

In [10]:
session_id = "kris'_session"
# Query
query = input("I'm UdaPlay the AI Researcher for Video Games. Ask me anything...")


out_path = Path(os.getcwd()) / f"{session_id}_udaplay_output.txt"
with open(out_path,"w",encoding = "utf-8") as file:
    file.write(f"User's {session_id} session with UdaPlay\n\n")
    i+=1
    run:Run = udaplay_agent.invoke(query, session_id)
    messages = run.get_final_state()["messages"]
    start = 0
    if i > 0:
        for j, m in enumerate(messages):
            if isinstance(m, UserMessage):
                start = j
        print("Omitting history\n")
        file.write("Omitting history\n")
    print_and_record_messages(messages[start:],file)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
Retrieving documents
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
Evaluating retrieved documents
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
Web searching
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
Evaluating retrieved documents
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Omitting history

__________________________________________________
User: When was the first Civilization game released and on what platform?



Retrieving documents from vector store



Retrieved documents are Not Useful.
The retrieved documents do not contain any information related to the Civilization game series. They focus on other video games such as Pokémon, Minecr